# iterative-probe — 反復の推移

`analysis/iterative-probe/results/epoch_metrics.csv` を読んで、global と hidden cohort の
AUROC / bACC、および GroupDRO の adversarial weight `q` の推移を見る。

表を先に出し、そのあと同じ数字を図にする。CSV が無い場合は先に集計する。

```bash
uv run python analysis/iterative-probe/collect.py <run-id>
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS = Path("results") if Path("results").is_dir() else Path("analysis/iterative-probe/results")
FIGURES = RESULTS.parent / "figures"

# dataviz の categorical slot 1 / 2。検証済みの組み合わせで、他の色に差し替えない。
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#dcdcd8"

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "axes.edgecolor": GRID,
        "axes.labelcolor": MUTED,
        "axes.titlecolor": INK,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "font.size": 9,
    }
)

frame = pd.read_csv(RESULTS / "epoch_metrics.csv")
frame[["stage", "stage_index", "run_epoch", "epoch"]]

## global と hidden の AUROC / bACC

`hidden_*` は cohort が存在する stage にだけある。warmup は hidden cohort を持たないので空になる。

**注意**: cohort は stage ごとに引き直されるため、stage01 と stage02 の hidden group は別物である。
stage をまたいだ `hidden_min_auroc` の差を改善として読むことはできない。

In [ ]:
COLUMNS = {
    "val/auroc": "global AUROC",
    "val/hidden_min_auroc": "hidden min AUROC",
    "val/bacc": "global bACC",
    "val/hidden_min_bacc": "hidden min bACC",
    "val/hidden_auroc_gap": "hidden AUROC gap",
}

table = frame.set_index([frame["stage"], frame["epoch"].astype(int)])[list(COLUMNS)].rename(columns=COLUMNS)
table.index.names = ["stage", "epoch"]
table.round(4)

In [ ]:
def stage_bands(axis: plt.Axes, data: pd.DataFrame) -> None:
    """stage の境目に区切り線を引き、上端に stage 名を置く。

    x 軸は run 全体の通し epoch なので、境目を示さないと stage が読めない。

    Args:
        axis: 描画先
        data: `stage` と `run_epoch` を持つ epoch 単位の表

    Returns:
        None
    """
    for _, group in data.groupby("stage_index"):
        left = group["run_epoch"].min()
        if left > 0:
            axis.axvline(left - 0.5, color=GRID, linewidth=1, zorder=0)
        axis.text(left, 1.02, group["stage"].iloc[0], transform=axis.get_xaxis_transform(), color=MUTED, fontsize=8, ha="left", va="bottom")


def series(axis: plt.Axes, data: pd.DataFrame, column: str, color: str, label: str) -> None:
    """1 系列を描き、最終点に系列名を直接添える。

    Args:
        axis: 描画先
        data: epoch 単位の表
        column: 描く列
        color: 系列の色
        label: 凡例と直接ラベルに使う名前

    Returns:
        None
    """
    values = data[column]
    axis.plot(data["run_epoch"], values, color=color, linewidth=2, marker="o", markersize=7, markeredgecolor="#fcfcfb", markeredgewidth=2, label=label, zorder=3)
    valid = values.dropna()
    if not valid.empty:
        last = valid.index[-1]
        axis.annotate(f"{valid.iloc[-1]:.3f}", (data.loc[last, "run_epoch"], valid.iloc[-1]), textcoords="offset points", xytext=(8, 0), va="center", color=MUTED, fontsize=8)


figure, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharex=True)
for axis, (metric, title) in zip(axes, [("auroc", "AUROC"), ("bacc", "bACC")], strict=True):
    series(axis, frame, f"val/{metric}", BLUE, "global")
    series(axis, frame, f"val/hidden_min_{metric}", ORANGE, "hidden min")
    stage_bands(axis, frame)
    axis.set_title(title, loc="left", fontsize=10, pad=20)
    axis.set_xlabel("run epoch")
    axis.set_xticks(frame["run_epoch"])
    # 系列が枠外へ出ないよう、その panel の実データから範囲を決める。
    shown = frame[[f"val/{metric}", f"val/hidden_min_{metric}"]].to_numpy(dtype=float)
    low, high = np.nanmin(shown), np.nanmax(shown)
    margin = max((high - low) * 0.15, 0.01)
    axis.set_ylim(low - margin, high + margin)
    axis.grid(axis="y", color=GRID, linewidth=0.8)
    axis.set_axisbelow(True)
    for side in ("top", "right"):
        axis.spines[side].set_visible(False)
    axis.legend(frameon=False, loc="lower left", fontsize=8, labelcolor=MUTED)

figure.tight_layout()
FIGURES.mkdir(exist_ok=True)
figure.savefig(FIGURES / "auroc_bacc.png", bbox_inches="tight")

## GroupDRO の adversarial weight

`q` は 10 cohort 分あるが、ここで見たいのは個々の cohort ではなく **一様分布からどれだけ離れたか**
なので、10 本を同じ色で重ねて基準線との距離を読む。`weight_entropy` の上限は一様分布の log(10)。

In [ ]:
clusters = [column for column in frame.columns if column.startswith("train/group_dro/q_")]
active = frame.dropna(subset=["train/group_dro/weight_entropy"])
uniform = 1 / len(clusters)

figure, axes = plt.subplots(1, 2, figsize=(10, 3.6))

axes[0].axhline(uniform, color=MUTED, linewidth=1, linestyle="--", zorder=1)
axes[0].text(active["run_epoch"].max(), uniform, f"  uniform {uniform:.2f}", color=MUTED, fontsize=8, va="center")
for column in clusters:
    axes[0].plot(active["run_epoch"], active[column], color=BLUE, linewidth=2, alpha=0.45, zorder=2)
axes[0].plot(active["run_epoch"], active["train/group_dro/max_q"], color=ORANGE, linewidth=2, marker="o", markersize=7, markeredgecolor="#fcfcfb", markeredgewidth=2, label="max q", zorder=3)
axes[0].plot([], [], color=BLUE, linewidth=2, alpha=0.45, label=f"q per cohort ({len(clusters)})")
axes[0].set_title("adversarial weight q", loc="left", fontsize=10, pad=20)
axes[0].set_ylim(0.08, 0.12)

entropy = active["train/group_dro/weight_entropy"]
ceiling = np.log(len(clusters))
axes[1].axhline(ceiling, color=MUTED, linewidth=1, linestyle="--", zorder=1)
axes[1].text(active["run_epoch"].max(), ceiling, f"  log(10) = {ceiling:.4f}", color=MUTED, fontsize=8, va="center")
axes[1].plot(active["run_epoch"], entropy, color=BLUE, linewidth=2, marker="o", markersize=7, markeredgecolor="#fcfcfb", markeredgewidth=2, label="weight entropy", zorder=3)
axes[1].set_title("weight entropy", loc="left", fontsize=10, pad=20)
# 変動幅は 0.003 しかない。軸を張り付かせると誤差が大きな動きに見えるので、
# 「1 cohort だけ重みが倍になった」程度の差（約 0.05）が見える範囲を取る。
axes[1].set_ylim(ceiling - 0.06, ceiling + 0.006)

for axis in axes:
    stage_bands(axis, active)
    axis.set_xlabel("run epoch")
    axis.set_xticks(active["run_epoch"])
    axis.grid(axis="y", color=GRID, linewidth=0.8)
    axis.set_axisbelow(True)
    for side in ("top", "right"):
        axis.spines[side].set_visible(False)
    axis.legend(frameon=False, loc="lower left", fontsize=8, labelcolor=MUTED)

figure.tight_layout()
figure.savefig(FIGURES / "group_dro_q.png", bbox_inches="tight")

In [ ]:
summary = active[["stage", "epoch"] + clusters + ["train/group_dro/max_q", "train/group_dro/weight_entropy"]]
summary = summary.set_index(["stage", summary["epoch"].astype(int)]).drop(columns="epoch")
summary.index.names = ["stage", "epoch"]
summary.round(4)